In [1]:
import pandas as pd

In [2]:
from nsepy import get_history as gh
import datetime as dt

In [3]:
stk_data = pd.read_csv("Tatacoffee13_21.csv")

In [4]:
stk_data=stk_data[["Open","High","Low","Close"]]

In [5]:
column="Close"

In [6]:
from sklearn.preprocessing import MinMaxScaler
Ms = MinMaxScaler()
data1= Ms.fit_transform(stk_data)
print("Len:",data1.shape)

Len: (2225, 4)


In [7]:
data1=pd.DataFrame(data1,columns=["Open","High","Low","Close"])

In [8]:
training_size = round(len(data1 ) * 0.80)
print(training_size)
X_train=data1[:training_size]
X_test=data1[training_size:]
print("X_train length:",X_train.shape)
print("X_test length:",X_test.shape)
y_train=data1[:training_size]
y_test=data1[training_size:]
print("y_train length:",y_train.shape)
print("y_test length:",y_test.shape)

1780
X_train length: (1780, 4)
X_test length: (445, 4)
y_train length: (1780, 4)
y_test length: (445, 4)


In [9]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
performance={"Model":[],"RMSE":[],"MaPe":[],"Lag":[],"Test":[]}

In [14]:
def combination_hwes(dataset, column):
    print("Running HWES for:", column)
    series = dataset[column]
    test_obs = 28
    train = series[:-test_obs]
    test = series[-test_obs:]

    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    import pandas as pd
    import numpy as np
    from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

    # Fit Holt-Winters model
    model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12)
    result = model.fit()

    # Forecast
    pred = result.forecast(steps=test_obs)
    preds = pd.DataFrame(pred, columns=[column])
    preds.to_csv("hwes_forecasted_{}.csv".format(test_obs))

    # Evaluate
    rmse = np.sqrt(mean_squared_error(test, pred))
    mape = mean_absolute_percentage_error(test, pred)

    performance["Model"].append(column + "_HWES")
    performance["RMSE"].append(rmse)
    performance["MaPe"].append(mape)
    performance["Lag"].append("Trend+Seasonality")
    performance["Test"].append(test_obs)

    perf = pd.DataFrame(performance)
    return perf, result, pred

In [15]:
perf,result,pred=combination_hwes(data1,column)

Running HWES for: Close


In [16]:
data1

,Open,High,Low,Close
0,0.849900,0.845408,0.856100,0.854203
1,0.856394,0.967399,0.861040,0.974481
2,0.988480,0.996439,0.984959,0.986240
3,0.985483,0.968105,0.960760,0.956749
4,0.955669,0.975319,0.955033,0.967132
...,...,...,...,...
2220,0.095842,0.096329,0.096510,0.097323
2221,0.097777,0.095745,0.096951,0.096041
2222,0.096466,0.093934,0.095252,0.094821
2223,0.094031,0.105047,0.093143,0.105673


In [17]:
perf

,Model,RMSE,MaPe,Lag,Test
0,Close_HWES,0.008355,0.076762,Trend+Seasonality,28


Holt-Winters, also known as Triple Exponential Smoothing, is an extension of simple exponential smoothing (SES).

It’s used for time series data with both trend and seasonality — like:

Monthly sales

Quarterly revenue

Daily temperature (seasonal patterns)

📈 It captures 3 components:
Component	Meaning	Example
Level	Average value over time	Average monthly sales
Trend	Increasing/decreasing direction	Sales growing each month
Seasonality	Repeating short-term cycle	Higher sales every December